In [1]:
# Top cell: install the libraries this notebook uses.
# If they are already installed, pip will simply say so.
%pip install -q pandas matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
CH = ROOT / 'charts'
CH.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150})
print('Paths ready:', PROC.exists(), CH.exists())

Paths ready: True True


# 1. Global temperature, by decade

First, a plain word guide. A temperature anomaly is not the temperature itself. It is the difference from a reference average. Here the reference is 1951 to 1980, the baseline NASA GISTEMP uses. So an anomaly of plus 1.0 means 1.0 degrees Celsius warmer than that 1951 to 1980 average. A negative anomaly means cooler than that average.

Source: NASA GISTEMP, file GLB.Ts+dSST.csv from the original climate_change repo, 1880 to 2023. The 2020s row in this file covers 2020 to 2023 only. It is labelled as partial everywhere it appears.

In [3]:
g = pd.read_csv(PROC / 'global_annual.csv')
print(g.head())
print(g.tail())
print('Years:', g['year'].min(), 'to', g['year'].max(), '| rows:', len(g))
assert len(g) == 144, 'expected 144 years, 1880 to 2023'

   year  annual_anomaly_c  winter_djf_c  spring_mam_c  summer_jja_c  \
0  1880             -0.17           NaN         -0.12         -0.16   
1  1881             -0.09         -0.17          0.05         -0.08   
2  1882             -0.11          0.08         -0.09         -0.15   
3  1883             -0.17         -0.34         -0.16         -0.09   
4  1884             -0.28         -0.10         -0.37         -0.31   

   autumn_son_c  
0         -0.19  
1         -0.19  
2         -0.18  
3         -0.18  
4         -0.28  
     year  annual_anomaly_c  winter_djf_c  spring_mam_c  summer_jja_c  \
139  2019              0.98          0.93          1.01          0.93   
140  2020              1.01          1.17          1.10          0.89   
141  2021              0.85          0.75          0.81          0.86   
142  2022              0.89          0.89          0.91          0.94   
143  2023              1.17          0.88          1.05          1.15   

     autumn_son_c  
139   

In [4]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(g['year'], g['annual_anomaly_c'], color='#1f77b4')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('Global temperature anomaly, each year from 1880 to 2023')
ax.set_xlabel('Year')
ax.set_ylabel('Anomaly in degrees Celsius, versus 1951-1980')
fig.savefig(CH / '01_global_annual_anomaly.png', bbox_inches='tight')
plt.show()
print('Warmest year in this file:', g.loc[g['annual_anomaly_c'].idxmax(), 'year'], g['annual_anomaly_c'].max())

Warmest year in this file: 2023 1.17


In [5]:
d = pd.read_csv(PROC / 'decade_means.csv')
print(d.to_string(index=False))

                decade  mean_anomaly_c  years_included  change_from_previous_decade_c
                 1880s          -0.209              10                            NaN
                 1890s          -0.235              10                         -0.026
                 1900s          -0.313              10                         -0.078
                 1910s          -0.326              10                         -0.013
                 1920s          -0.238              10                          0.088
                 1930s          -0.122              10                          0.116
                 1940s           0.042              10                          0.164
                 1950s          -0.047              10                         -0.089
                 1960s          -0.031              10                          0.016
                 1970s           0.035              10                          0.066
                 1980s           0.247              10

In [6]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(d['decade'], d['mean_anomaly_c'], color='#2ca02c')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('Average temperature anomaly by decade')
ax.set_xlabel('Decade (2020s covers 2020-2023 only)')
ax.set_ylabel('Mean anomaly in degrees Celsius')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.savefig(CH / '02_decade_means.png', bbox_inches='tight')
plt.show()
print('1980s mean:', round(d[d['decade']=='1980s']['mean_anomaly_c'].iloc[0], 3))
print('2010s mean:', round(d[d['decade']=='2010s']['mean_anomaly_c'].iloc[0], 3))

1980s mean: 0.247
2010s mean: 0.806


## What this shows

The 1880s average was minus 0.209 degrees. The 2010s average was plus 0.806 degrees. The gap between those two decade averages is about 1.02 degrees. Single years still jump up and down, but the decade averages climb, step by step, from the 1980s onward. That is why this project uses decade averages as well as single years: they smooth out one hot or cold year.